# 第 08 章　KNN、支持向量机与浅层神经网络

> **本章问题**：KNN、SVR 和 MLP 在同一片留出区域上表现如何？误差和耗时要怎样一起看？

这次我们把三种思路放在一起比较：KNN 去找特征相似的样本，SVR 尝试在误差带内拟合，MLP 用多层变换表示更复杂的关系。它们都对特征尺度和样本量比较敏感，地理空间的分布也会影响结果。

学完本章，你可以分清回归和分类的模型接口，理解 SVR 的几个关键参数，并在相同空间折和测试区上比较模型。


## 8.1　先分清：我们做的是分类还是回归？

LST 是连续数值，所以这里要用回归器。`SVC` 输出类别，`SVR` 输出实数；KNN 和 MLP 也各自有分类与回归接口。本章全部使用回归版本。

下面这张表总结了各模型的大致思路和对特征尺度的敏感程度。KNN、SVR、MLP 都很在意尺度差异，所以我们把填补和标准化放进同一条 Pipeline，并且只在训练折里拟合。


In [ ]:

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_models as gm
import geoml_style as gs

gs.setup()
np.random.seed(gm.SEED)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

print("Python:", sys.executable)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))
print("项目目录:", PROJECT)


## 8.2　沿用固定划分，测试区继续锁好

第 04 章已经固定 5 个空间折和中南部独立测试区。本章不重新划分，也不拿测试区选参数。`C`、`gamma` 和 `k` 都只根据训练区的空间交叉验证来选。

下面先确认样本数：训练区 5796 个像元，测试区 1360 个。


In [ ]:

df = gd.load_splits()
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()
features = gm.FEATURES

print(f"训练区：{len(train)} 行，{train['fold'].nunique()} 个空间折")
print(f"独立测试区：{len(test)} 行")
print("训练区目标均值 / 标准差：", round(train["lst"].mean(), 3), "/", round(train["lst"].std(), 3))
print("测试区目标均值 / 标准差：", round(test["lst"].mean(), 3), "/", round(test["lst"].std(), 3))
display(train[["row", "col", "lst", *features]].head())


## 8.3　`C`、`gamma`、`epsilon` 各管什么？

SVR 会在预测函数两边留出一条“不敏感管”，宽度是 `2 × epsilon`。落在管内的误差暂时不计，超出管道的部分才开始受罚。

- **`epsilon`** 是管道半宽。调大以后模型更宽容，可能变得太简单；调小则更努力贴近样本。
- **`C`** 控制管外误差的惩罚。调大时模型更努力减少训练误差，也更容易过拟合；调小则正则化更强。
- **`gamma`** 只用于 RBF 核。它决定一个样本影响多大范围，值大时影响更局部，值小时曲面更平滑。

线性核没有 `gamma`。下面先固定 `epsilon=0.2`，只用训练区空间折比较 `C` 和 `gamma`。


## 8.4　用空间折扫描 RBF-SVR 参数

这里试 4×4 组参数。网格刻意保持小一些，好让我们看清参数怎么影响结果，不是要穷举全世界的组合。每组都用同一套空间折打分，测试区还没有参与。


In [ ]:

def scaled(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

C_GRID = [0.1, 1.0, 10.0, 100.0]
GAMMA_GRID = [0.01, 0.03, 0.1, 0.3]
EPSILON = 0.2

rows = []
for C in C_GRID:
    for gamma in GAMMA_GRID:
        folds = gm.cv_scores(
            lambda C=C, gamma=gamma: scaled(
                SVR(kernel="rbf", C=C, gamma=gamma, epsilon=EPSILON)
            ),
            train,
        )
        rows.append({
            "C": C, "gamma": gamma, "epsilon": EPSILON,
            "RMSE_折均": folds["RMSE"].mean(),
            "RMSE_折标准差": folds["RMSE"].std(ddof=1),
            "MAE_折均": folds["MAE"].mean(),
            "R2_折均": folds["R2"].mean(),
        })
        print(f"C={C:>5g}  gamma={gamma:>4g}  CV RMSE={folds['RMSE'].mean():.4f}", flush=True)

svr_grid = pd.DataFrame(rows).sort_values("RMSE_折均").reset_index(drop=True)
svr_grid.to_csv(gd.TABLES / "08_svr_grid.csv", index=False, encoding="utf-8-sig")
best = svr_grid.iloc[0]
print("\n训练区 CV 最优组合：")
display(best.to_frame().T)


In [ ]:

pivot = svr_grid.pivot(index="gamma", columns="C", values="RMSE_折均").sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(6.8, 4.5), constrained_layout=True)
im = ax.imshow(pivot.values, cmap="YlGnBu_r", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c:g}" for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{g:g}" for g in pivot.index])
ax.set_xlabel("C（惩罚强度）")
ax.set_ylabel("gamma（RBF 影响范围）")
ax.set_title("RBF-SVR 参数扫描：训练区空间 CV RMSE")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center",
                fontsize=8.5, color="white" if pivot.values[i, j] < np.nanmedian(pivot.values) else "#222222")
best_i = list(pivot.index).index(best["gamma"])
best_j = list(pivot.columns).index(best["C"])
ax.add_patch(Rectangle((best_j - 0.5, best_i - 0.5), 1, 1, fill=False,
                       edgecolor="#b23a48", linewidth=2.2))
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cb.set_label("空间 CV RMSE（原始像元值）")
gs.save(fig, "08_svr_grid.png")
plt.show()

print(f"网格最优：C={best['C']:g}, gamma={best['gamma']:g}, epsilon={EPSILON:g}")
print(f"训练区 CV RMSE = {best['RMSE_折均']:.4f}（折间标准差 {best['RMSE_折标准差']:.4f}）")


看这张热图时，别只挑颜色最深的一个格子。先观察低 RMSE 有没有形成一片相邻区域，再看看各折之间的波动。如果只有孤零零一组参数最好，差异也可能只是某一折碰巧。

本章把训练区空间交叉验证选出的参数用于最终模型。接下来，我们让 KNN、线性 SVR、RBF-SVR 和 MLP 在同一套划分上比较。


## 8.5　四个模型，用同一把尺子比较

四个模型的设置如下：

- **KNN**：`k=9`，按距离加权。这里说的“近邻”是特征相似，不一定是地图上相邻。
- **线性 SVR**：`C=1.0`、`epsilon=0.2`，用来检验线性关系是否够用。
- **RBF-SVR**：沿用上一节选出的 `C`、`gamma`，`epsilon=0.2`。
- **MLP**：两层隐藏层 `(32, 16)`，开启早停并固定随机种子。

接下来每个模型都先跑 5 个空间折，再用整个训练区拟合一次并预测测试区。耗时只统计最后这次拟合和预测，不包含交叉验证，所以适合看相对差异，不适合当严格的性能基准。


In [ ]:

C_best = float(best["C"])
gamma_best = float(best["gamma"])

factories = {
    "KNN (k=9)": lambda: scaled(KNeighborsRegressor(n_neighbors=9, weights="distance")),
    "SVR 线性": lambda: scaled(SVR(kernel="linear", C=1.0, epsilon=EPSILON)),
    "SVR RBF": lambda: scaled(SVR(kernel="rbf", C=C_best, gamma=gamma_best, epsilon=EPSILON)),
    "MLP (32,16)": lambda: scaled(MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        max_iter=600,
        random_state=gm.SEED,
    )),
}

fold_tables = {}
test_predictions = {}
metric_rows = []
fitted_models = {}

for name, factory in factories.items():
    folds = gm.cv_scores(factory, train)
    fold_tables[name] = folds

    model = factory()
    t0 = time.perf_counter()
    model.fit(train[features], train["lst"])
    pred = model.predict(test[features])
    elapsed = time.perf_counter() - t0
    met = gm.metrics(test["lst"], pred)

    fitted_models[name] = model
    test_predictions[name] = pred
    metric_rows.append({
        "模型": name,
        "MAE_CV": folds["MAE"].mean(), "MAE_CV_标准差": folds["MAE"].std(ddof=1),
        "RMSE_CV": folds["RMSE"].mean(), "RMSE_CV_标准差": folds["RMSE"].std(ddof=1),
        "R2_CV": folds["R2"].mean(), "R2_CV_标准差": folds["R2"].std(ddof=1),
        "MAE_测试": met["MAE"], "RMSE_测试": met["RMSE"], "R2_测试": met["R2"],
        "耗时秒": elapsed,
    })
    print(f"{name:12s} CV RMSE={folds['RMSE'].mean():.4f}  "
          f"测试 RMSE={met['RMSE']:.4f}  R²={met['R2']:.3f}  用时={elapsed:.2f}s", flush=True)

metrics = pd.DataFrame(metric_rows).sort_values("RMSE_测试").reset_index(drop=True)
fold_scores = pd.concat([t.assign(模型=name) for name, t in fold_tables.items()], ignore_index=True)

metrics.to_csv(gd.TABLES / "08_metrics.csv", index=False, encoding="utf-8-sig")
fold_scores.to_csv(gd.TABLES / "08_fold_scores.csv", index=False, encoding="utf-8-sig")

pred_out = test[["row", "col", "x", "y", "lst"]].rename(columns={"lst": "obs"})
for name, pred in test_predictions.items():
    pred_out[f"pred_{name}"] = pred
pred_out.to_csv(gd.TABLES / "08_test_predictions.csv", index=False, encoding="utf-8-sig")

display(metrics.round(3))

gd.write_json({
    "notebook": "08_KNN_SVR_MLP.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "best_svr_params_on_training_cv": {
        "C": float(best["C"]), "gamma": float(best["gamma"]), "epsilon": EPSILON,
        "cv_rmse": float(best["RMSE_折均"]),
    },
    "models": metrics.round(6).to_dict(orient="records"),
    "note": "耗时为本机单次拟合+预测，仅作相对参考；测试区只评估一次。",
    "unverified": [
        "LST/降水/土壤水分单位未在元数据中注明，统一按原始像元值。",
        "LST 仅知 2021 年，与 2020 年降水/土壤水分不同期。",
        "案例仅作留出区域估计的方法演示，不是甘南县 LST 产品。",
    ],
}, "08_run_summary.json")


In [ ]:

fold_rmse = {
    name: fold_tables[name]["RMSE"].to_numpy()
    for name in metrics["模型"]
}
test_rmse = {
    row["模型"]: row["RMSE_测试"]
    for _, row in metrics.iterrows()
}
gs.model_comparison(
    fold_rmse,
    test_rmse,
    out_name="08_model_compare.png",
    title="KNN、SVR 与 MLP：空间折误差与独立测试区",
    ylabel="RMSE（原始像元值）",
)
print("图件：outputs/figures/08_model_compare.png")


In [ ]:

timing = metrics[["模型", "耗时秒"]].sort_values("耗时秒", ascending=True)
fig, ax = plt.subplots(figsize=(7.2, 3.5), constrained_layout=True)
bars = ax.barh(timing["模型"], timing["耗时秒"], color=[gs.PALETTE[i % len(gs.PALETTE)] for i in range(len(timing))])
for bar, val in zip(bars, timing["耗时秒"]):
    ax.text(val + max(timing["耗时秒"]) * 0.015, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f} s", va="center", fontsize=9)
ax.set_xlabel("拟合 + 预测耗时 / 秒")
ax.set_ylabel("")
ax.set_title("四个模型的本机单次耗时（仅作相对参考）")
ax.set_xlim(0, max(timing["耗时秒"]) * 1.18)
ax.grid(axis="x", color=gs.GRID, linewidth=0.6, alpha=0.8)
ax.set_axisbelow(True)
gs.save(fig, "08_timing.png")
plt.show()

print("耗时最长：", timing.iloc[-1]["模型"], f"{timing.iloc[-1]['耗时秒']:.2f}s")
print("耗时最短：", timing.iloc[0]["模型"], f"{timing.iloc[0]['耗时秒']:.2f}s")


先看误差，再看时间。一个模型即使快一些，如果误差明显更大，也未必划算；反过来，跑得慢很多却只改善一点点，也要想想值不值得。

这里的秒数来自本机的一次运行。换一台电脑会变，重点看相对关系，不要把它当作模型的固有属性。


## 8.6　把残差放回地图上看

RMSE 把所有误差压成一个数，地图能补充告诉我们误差落在什么位置。下面选取测试区表现较好的模型，画出观测、预测和残差。前两张共用色标，残差以 0 为中心，定义为“预测减观测”。

如果残差在地图上成片偏红或偏蓝，说明模型在这些区域有系统偏差，不只是零散的小误差。


In [ ]:

best_model_name = metrics.iloc[0]["模型"]
best_pred = test_predictions[best_model_name]
window = gd.test_window(df)
pred_grid = gd.grid_from_predictions(window, best_pred)
resid_grid = pred_grid - window["obs"]

from shapely.geometry import box
import geopandas as gpd

boundary = gpd.read_file(gd.RAW / "GanNan.shp")
gs.plot_triptych(
    window["obs"], pred_grid, resid_grid, window["extent"],
    title=f"{best_model_name}：独立测试区估计",
    out_name="08_triptych_best.png",
    cbar_label="LST（原始像元值）",
    cmap="magma",
    boundary=boundary,
    crs="EPSG:4326",
)
print("测试区表现最好的模型：", best_model_name)
print(f"测试 RMSE = {metrics.iloc[0]['RMSE_测试']:.4f}，MAE = {metrics.iloc[0]['MAE_测试']:.4f}")



这次运行出现了一个值得注意的“分裂”：

- **空间 CV**：SVR RBF 最低（RMSE 1.256，折间标准差 0.128）；MLP 为 1.335（SD 0.234）；KNN 1.337；线性 SVR 1.383。
- **独立测试区**：MLP 的 RMSE 1.493、MAE 1.186 最低；SVR RBF 为 1.506 / 1.253；KNN 1.606；线性 SVR 1.618。
- MLP 与 SVR RBF 的测试 RMSE 只差约 0.013，远小于空间折之间的波动；不能据此宣布 MLP 更优，只能说“在这个留出区上略低”。
- SVR RBF 更稳（折间 SD 0.128，MLP 0.234），也更快（约 1.7 s vs 5.7 s，本机单次）。如果只能选一个交付，SVR RBF 的性价比更好。
- 线性 SVR 与 RBF-SVR 的差距说明关系并非纯线性；但 RBF 的最优 `C` 落在搜索下界 0.1，提示还可以往更小的 `C` 试。本章不扩展网格，避免把“调参”变成对测试区的变相窥探。

## 8.7 怎么读这些结果

把表格和地图合起来，至少可以得到三层结论：

1. **误差**：在线性 SVR、RBF-SVR、KNN、MLP 中，谁在独立测试区的 RMSE/MAE 最低？谁的空间折波动最大？
2. **偏差**：测试区目标均值高于训练区。若模型预测整体偏低或残差成片偏蓝，这不是“随机噪声”，而是训练分布没有覆盖测试区造成的系统偏差。
3. **代价**：SVR 和 MLP 需要标准化，MLP 还需要选择迭代轮数和随机种子；KNN 几乎不用训练，但预测时要保留全部训练样本。地理教学里，“可解释、可复现、算得动”与“多 0.02 的 R²”同样重要。

注意：本章为了在有限课时内完成，只做了很小的 SVR 网格和固定的 KNN/MLP 配置。真实研究应把调参、特征选择和模型选择全部限制在训练区，并报告多次随机种子或嵌套 CV 的波动。


## 8.8　再从地理角度想一遍

- KNN 找的是特征空间里的近邻，不一定是地图上的近邻，也不会自动理解山脉或行政边界。
- 随机折容易让相邻像元分到训练和验证两边，分数可能偏乐观。本章继续使用空间折和整块测试区。
- 降水、土壤水分的原始像元比目标 LST 粗很多。重采样后变成较大的常值块，模型不能据此识别 1 km 尺度的细节。
- SVR 的核函数和 MLP 的权重描述模型如何预测，不等同于因果机制。
